In [1]:
import datasets

/opt/conda/envs/reasoning_sft/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = datasets.load_dataset("hkgc/math3to5_olympiads_aime", split="train")

Generating train split: 100%|██████████| 18328/18328 [00:00<00:00, 499572.41 examples/s]


In [3]:
ds[0]

{'problem': '## A2 ALB\n\n3) If $x^{3}-3 \\sqrt{3} x^{2}+9 x-3 \\sqrt{3}-64=0$, find the value of $x^{6}-8 x^{5}+13 x^{4}-5 x^{3}+49 x^{2}-137 x+2015$.\n',
 'solution_hint': '$x^{3}-3 \\sqrt{3} x^{2}+9 x-3 \\sqrt{3}-64=0 \\Leftrightarrow(x-\\sqrt{3})^{3}=64 \\Leftrightarrow(x-\\sqrt{3})=4 \\Leftrightarrow x-4=\\sqrt{3} \\Leftrightarrow x^{2}-8 x+16=3 \\Leftrightarrow$ $x^{2}-8 x+13=0$\n\n$x^{6}-8 x^{5}+13 x^{4}-5 x^{3}+49 x^{2}-137 x+2015=\\left(x^{2}-8 x+13\\right)\\left(x^{4}-5 x+9\\right)+1898=0+1898=1898$\n\n',
 'answer': '1898',
 'problem_type': 'Algebra',
 'source': 'olympiads',
 'level': None,
 'type': None,
 'solution': '\\(\\boxed{1898}\\)'}

In [4]:
samples = []
for index, item in enumerate(ds):
    prompt = [
        {"role": "system", "content": "Please reason step by step, and put your final answer within \\boxed{}."},
        {"role": "user", "content": item["problem"]},
    ]
    sample = {
        "data_source": "numina_math",
        "prompt": prompt,
        "ability": "math",
        "reward_model": {"ground_truth": item["answer"], "style": "rule"},
        "extra_info": {"index": index, "split": "train"},
        "problem": item["problem"],
        "answer": item["answer"],
        "solution": item["solution"],
    }
    samples.append(sample)

In [6]:
import pandas as pd

df = pd.DataFrame(samples)
df.to_parquet("data/train.parquet", index=False)
print(f"Saved {len(df)} samples to data/train.parquet")

Saved 18328 samples to data/train.parquet


In [9]:
import json

with open("./tmp/HINT/dataset/training_dataset/dapo-selected-30k.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

print(f"Total samples: {len(data)}")
print(f"Keys: {data[0].keys()}")
data[0]

Total samples: 27625
Keys: dict_keys(['question', 'answer', 'abstract_hint', 'medium_hint', 'direct_hint'])


{'question': 'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.',
 'answer': '34',
 'abstract_hint': 'Think about the properties of angle bisectors and the given trigonometric value. How might the angle bisector theorem and the Pythagorean theorem help you relate the sides of the triangle? What relationship can you infer from the given ratio $\\frac{BD}{CD} = \\frac{3}{2}$ and the right angle at $\\angle BDC$?',
 'medium_hint': 'First, identify the given values: $\\sin \\angle A = \\frac{4}{5}$, $AD = 1$, and $\\frac{BD}{CD} = \\frac{3}{2}$. Since $\\angle A < 90^\\circ$, you can determine $\\cos \\angle A$ using the Pythagorean identity. Next, c

In [10]:
samples = []
for index, item in enumerate(data):
    prompt = [
        {"role": "system", "content": "Please reason step by step, and put your final answer within \\boxed{}."},
        {"role": "user", "content": item["question"]},
    ]
    sample = {
        "data_source": "numina_math",
        "prompt": prompt,
        "ability": "math",
        "reward_model": {"ground_truth": item["answer"], "style": "rule"},
        "extra_info": {"index": index, "split": "train"},
        "problem": item["question"],
        "answer": item["answer"],
        "solution": item["direct_hint"],
    }
    samples.append(sample)

In [11]:
df = pd.DataFrame(samples)
df.to_parquet("data/train_hint.parquet", index=False)
print(f"Saved {len(df)} samples to data/train_hint.parquet")

Saved 27625 samples to data/train_hint.parquet


In [14]:
df = pd.read_parquet("imperfect_trace/deepseek_results.parquet")
df

,index,data_source,problem,answer,reasoning,content,full_responses,scores,num_correct,accuracy
0,18,numina_math,Question 62: Given positive real numbers $a$ a...,54,"[We are given: ""Question 62: Given positive re...","[问题 62: 已知正实数 $a$ 和 $b$ 满足 $a(a+b)=27$, 试求 $a^...","[We are given: ""Question 62: Given positive re...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",0.0,0.00
1,28,numina_math,"Question 1. Let $z \in C, |z|=1$ but $z \neq -...",z=\frac{1+t i}{1-t i},"[We are given: ""Question 1. Let $z \in C, |z|=...",[For any complex number \(z\) on the unit circ...,"[We are given: ""Question 1. Let $z \in C, |z|=...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",0.0,0.00
2,31,numina_math,Find the number of integer solutions of the eq...,0,"[, , , , , , , ]",[We are given the equation:\n\[\nx^{2016} + (2...,[We are given the equation:\n\[\nx^{2016} + (2...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",0.0,0.00
3,33,numina_math,"## Task 4 - 070914\n\nFour teams $A, B, C$, an...",4,"[, , , , , , , ]","[We are given: Four teams A, B, C, D. Each tea...","[We are given: Four teams A, B, C, D. Each tea...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",0.0,0.00
4,44,numina_math,8. If the line $y=x$ is tangent to the curve $...,1 \text{ or } \frac{13}{4},"[, , , , , , , ]","[We are given: ""If the line $y=x$ is tangent t...","[We are given: ""If the line $y=x$ is tangent t...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",0.0,0.00
...,...,...,...,...,...,...,...,...,...,...
108,973,numina_math,Example 2 After a round-robin volleyball tourn...,7,"[, , , , , , , ]","[We are given: ""After a round-robin volleyball...","[We are given: ""After a round-robin volleyball...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",0.0,0.00
109,974,numina_math,2. The range of $a$ for which the equation $\s...,D,"[, , , , , , , ]","[We are given: ""The range of $a$ for which the...","[We are given: ""The range of $a$ for which the...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",0.0,0.00
110,988,numina_math,"Tom throws a football to Wes, who is a distanc...",\textbf{(E)},"[, , , , , , , ]","[We are given: ""Tom throws a football to Wes, ...","[We are given: ""Tom throws a football to Wes, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",0.0,0.00
111,990,numina_math,12. Simplify: $\sum_{k=1}^{n} \frac{\cos 3^{k}...,\frac{1}{2}(3\cotx-\frac{\cot3^{n}x}{3^{n-1}}),"[, , , , , , , ]",[We are given: \nSimplify: $\sum_{k=1}^{n} \fr...,[We are given: \nSimplify: $\sum_{k=1}^{n} \fr...,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0]",2.0,0.25


In [15]:
df.iloc[0]["full_responses"]

array(['We are given: "Question 62: Given positive real numbers $a$ and $b$ satisfy $a(a+b)=27$, try to find the maximum value of $a^{2} b$."\n\nWe need to translate this into Chinese. The instruction says: "保留源文本的换行和格式，直接输出翻译结果。 (注意：最后一句是解释说明，不是原文的一部分，因此在最终输出时应去掉。)"\n\nSo we need to output the translation exactly preserving line breaks and format. The note says the last sentence is an explanation and not part of the original text, so we should omit it in the final output.\n\nOriginal text: "Question 62: Given positive real numbers $a$ and $b$ satisfy $a(a+b)=27$, try to find the maximum value of $a^{2} b$."\n\nWe translate: "问题 62: 已知正实数 $a$ 和 $b$ 满足 $a(a+b)=27$, 试求 $a^{2} b$ 的最大值。"\n\nWe should output exactly that, preserving the LaTeX math mode.\n\nThus final output: \n问题 62: 已知正实数 $a$ 和 $b$ 满足 $a(a+b)=27$, 试求 $a^{2} b$ 的最大值。</think>问题 62: 已知正实数 $a$ 和 $b$ 满足 $a(a+b)=27$, 试求 $a^{2} b$ 的最大值。',
       'We are given: "Question 62: Given positive real numbers $a$ and $b$ satisfy $a(a+b)=

In [20]:
len(df.iloc[0]["full_responses"][0].lower().split("but"))

1